# 3D Gaussian Splatting with GSplat in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varunm18/diffusion-time-machine/blob/varun/notebooks/colab_gsplat_test.ipynb)

This notebook runs the **CUDA-accelerated 3D Gaussian Splatting (`gsplat`)** pipeline using images stored in your Google Drive at `MyDrive/Time-Machine/riverview`.

### Pipeline Stages:
1. **Pose Estimation & Point Cloud**: `mini-DUSt3R` extracts camera intrinsics, camera extrinsics (`transforms.json`), and sparse point cloud (`sparse/0/points3D.ply`).
2. **3DGS Optimization**: `gsplat` trains 3D Gaussians on CUDA GPU using rasterization and adaptive density control.
3. **Export**: Outputs both standard `scene_3dgs.ply` and lightweight `scene_3dgs.splat` directly to your Google Drive.

## 1. Mount Google Drive
Mount your Google Drive to access `MyDrive/Time-Machine/riverview`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

IMAGE_DIR = Path('/content/drive/MyDrive/Time-Machine/riverview')
assert IMAGE_DIR.exists(), f'Image directory not found at {IMAGE_DIR}. Please check the path in your Google Drive.'

image_files = sorted([f for f in IMAGE_DIR.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Found {len(image_files)} images in {IMAGE_DIR}:')
for img in image_files[:5]:
    print(f'  - {img.name}')
if len(image_files) > 5:
    print(f'  ... and {len(image_files) - 5} more.')

## 2. Check GPU Hardware Acceleration
Verify that a CUDA-capable GPU (T4, V100, A100, or L4) is assigned to this runtime.
*If GPU is not available, go to **Runtime > Change runtime type > GPU**.*

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'CUDA GPU is required for gsplat. Please enable GPU in Runtime settings.'
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')
print(f'PyTorch Version: {torch.__version__}')

## 3. Clone Repository & Install Dependencies
Clone the `diffusion-time-machine` repo (branch `varun`) and install `gsplat` alongside project requirements.

In [ ]:
import os

# Clone repository if not already cloned
if not os.path.exists('/content/diffusion-time-machine'):
    !git clone -b varun https://github.com/varunm18/diffusion-time-machine.git /content/diffusion-time-machine

%cd /content/diffusion-time-machine
!git pull origin varun

# Install project requirements and CUDA gsplat
!pip install -q -r requirements.txt
!pip install -q gsplat

In [ ]:
# Verify library imports
import gsplat
from src.viz import DUST3R, GSplat, visualize

print(f'gsplat version: {gsplat.__version__}')
print('src.viz components (DUST3R, GSplat, visualize) loaded successfully!')

## 4. Set Workspace Directory
Configure the destination workspace folder where camera poses, point clouds, and 3DGS models will be saved.
Saving to Google Drive ensures your trained 3D models persist across Colab sessions.

In [ ]:
# Store outputs in Google Drive so they persist
WORKSPACE = Path('/content/drive/MyDrive/Time-Machine/riverview_workspace')
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f'Input images : {IMAGE_DIR}')
print(f'Output folder: {WORKSPACE}')

## 5. Step 1: mini-DUSt3R Reconstruction
Extracts camera intrinsics, camera extrinsics (`transforms.json`), and sparse point cloud (`sparse/0/points3D.ply`).

In [ ]:
# Run mini-DUSt3R camera estimation & initial point cloud on GPU
dust3r = DUST3R(device='cuda')
results = dust3r.run(image_dir=IMAGE_DIR, output_workspace=WORKSPACE)

print(f'Successfully saved camera poses to {WORKSPACE / "transforms.json"}')
print(f'Point cloud contains {len(results.point_cloud.vertices):,} points.')

## 6. Step 2: 3D Gaussian Splatting Training (`gsplat`)
Trains 3D Gaussian Splats on CUDA GPU using `gsplat` rasterization and adaptive density control.

In [ ]:
# Set training iterations (7000 is standard for full quality; 2000-3000 for quick testing)
TRAIN_STEPS = 7000

splatter = GSplat(iterations=TRAIN_STEPS, device='cuda')
output_ply = splatter.train(workspace_dir=WORKSPACE, callback_every=500)

print('3DGS training complete!')
print(f'Saved PLY   : {output_ply}')
print(f'Saved Splat : {output_ply.with_suffix(".splat")}')

## 7. Alternative: Single Command CLI Runner
You can also run the entire end-to-end pipeline in one shell command using `scripts/run_3d_pipeline.py`:

In [ ]:
!python scripts/run_3d_pipeline.py \
  --image_dir "/content/drive/MyDrive/Time-Machine/riverview" \
  --workspace "/content/drive/MyDrive/Time-Machine/riverview_workspace" \
  --device cuda \
  --gsplat

## 8. Check Results & Inspect Files
Inspect the exported files. Because `WORKSPACE` is in Google Drive, they are already safely saved to your Google Drive!

In [ ]:
ply_file = WORKSPACE / 'scene_3dgs.ply'
splat_file = WORKSPACE / 'scene_3dgs.splat'
transforms_file = WORKSPACE / 'transforms.json'

for f in [ply_file, splat_file, transforms_file]:
    if f.exists():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'{f.name:<20}: {size_mb:.2f} MB  ({f})')
    else:
        print(f'{f.name:<20}: NOT FOUND')

## 9. How to View Your 3D Gaussian Splat
You have multiple options to view the generated scene:

### Option A: Online Web Splat Viewer (Instant in Browser)
1. Download `scene_3dgs.splat` or locate it in your Google Drive folder.
2. Open [SuperSplat Viewer](https://playcanvas.com/supersplat/editor) or [Antimatter15 Splat Viewer](https://antimatter15.com/splat/).
3. Drag and drop `scene_3dgs.splat` (or `scene_3dgs.ply`) directly into your browser!

### Option B: Local Interactive Rerun Viewer
On your local machine (Mac or PC with repo cloned):
```bash
python -m src.viz.viewer --target /path/to/riverview_workspace
```